In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from plottr.data.datadict_storage import datadict_from_hdf5
import lmfit

In [9]:
from scipy import fftpack

## FFT

In [2]:
def FFT(x, y):
    off_ini = np.mean(y)
    y_mod = y - off_ini
    Y = np.abs(np.fft.fft(y_mod))
    dt = 2e-9
    N = len(y)
    t = x
    f = np.fft.fftfreq(N, dt)
    sorted_idx=np.argsort(f)
    x_fft = f[sorted_idx]
    y_fft = Y[sorted_idx]
    return x_fft, y_fft

In [26]:
def FFT_show(data):
    header = "D:/K_sunada/result/CDY152/"
    dd = datadict_from_hdf5(header+data+"/data")
    x = dd['time']['values']
    y = dd['waveform']['values'] - dd['waveform1']['values'] 
    readout_freq = 10.306e9
    readout_if_freq = (10.423-10.306)*1e9
    readout_lo_freq = readout_freq + readout_if_freq

    x_fft = FFT(x, y)[0]
    y_fft = FFT(x, y)[1]

    fig, (ax1, ax2) = plt.subplots(nrows=2)
    ax1.set_xlabel('Time (ns)')
    ax1.set_ylabel('Waveform (V)')
    ax1.plot(x,  y)
    ax1.plot(x, np.abs(signal.hilbert(y)))
    ax1.tick_params(axis="x", direction="in")
    ax1.tick_params(axis="y", direction="in")

    ax2.set_xlabel('Frequency(Hz)')
    ax2.set_ylabel('Power (arb. u.)')
    ax2.plot(readout_lo_freq-x_fft, y_fft)
    
    ax2.tick_params(axis="x", direction="in")
    ax2.tick_params(axis="y", direction="in")
    # ax2.set_xlim(0, 250e6)
    ax2.set_xlim(readout_lo_freq-250e6, readout_lo_freq)
    peak = readout_lo_freq-x_fft[signal.argrelmax(y_fft, order=10)]

    plt.show()
    return peak


In [14]:
def FFT_show_without_env(data):
    header = "D:/K_sunada/result/CDY152/"
    dd = datadict_from_hdf5(header+data+"/data")
    x = dd['time']['values']
    y = dd['waveform']['values']  - dd['waveform2']['values']
    readout_freq = 10.306e9
    readout_if_freq = (10.423-10.306)*1e9
    readout_lo_freq = readout_freq + readout_if_freq

    x_fft = FFT(x, y)[0]
    y_fft = FFT(x, y)[1]

    ifft_time = np.abs(fftpack.ifft(FFT(x, y)))

    fig, (ax1, ax2) = plt.subplots(nrows=2)
    ax1.set_xlabel('Time (ns)')
    ax1.set_ylabel('Waveform (V)')
    ax1.plot(x,  y)
    ax1.plot(ifft_time[0], ifft_time[1])
    ax1.tick_params(axis="x", direction="in")
    ax1.tick_params(axis="y", direction="in")

    ax2.set_xlabel('Frequency(Hz)')
    ax2.set_ylabel('Power (arb. u.)')
    ax2.plot(readout_lo_freq-x_fft, y_fft)
    
    ax2.tick_params(axis="x", direction="in")
    ax2.tick_params(axis="y", direction="in")
    # ax2.set_xlim(0, 250e6)
    ax2.set_xlim(readout_lo_freq-250e6, readout_lo_freq)
    peak = readout_lo_freq-x_fft[signal.argrelmax(y_fft, order=10)]

    plt.show()
    return peak


In [5]:
def waveform_show(data):
    header = "D:/K_sunada/result/CDY152/"
    dd = datadict_from_hdf5(header+data+"/data")
    x = dd['time']['values']
    y = dd['waveform']['values'] - dd['waveform1']['values']


    fig, (ax1) = plt.subplots(nrows=1)
    ax1.set_xlabel('Time (ns)')
    ax1.set_ylabel('Waveform (V)')
    ax1.plot(x,  y)
    ax1.tick_params(axis="x", direction="in")
    ax1.tick_params(axis="y", direction="in")
    plt.tick_params(
                top='on',
                bottom='on`',
                left='on',
                right='on',
               )
    plt.show()

In [ ]:
def lowpass(x, samplerate, fp, fs, gpass, gstop):
    fn = samplerate/2
    wp = fp/fn
    ws = fs/fn
    N, Wn = signal.buttord(wp, ws, gpass, gstop)
    b, a = signal.butter(N, Wn, "low")
    y = signal.filtfilt(b, a, x)
    return y

In [ ]:
y0low = lowpass( y0*np.exp(-1j*1.14e8*x*1e-9*2*np.pi), 2e-9, -0.1e8, 0.11e8, 2, 40)
y0lowfft = FFT(x, y0low )[1]

fig, (ax01, ax02) = plt.subplots(nrows=2)
ax01.set_xlabel('time(ns)')
ax01.set_ylabel('y')
ax01.plot(x, y0)
ax01.plot(x, -2* y0low)

ax02.set_xlabel('frequency(Hz)')
ax02.set_ylabel('Y/|N|')
# ax02.set_xlim(10.28e9, 10.38e9)

ax02.plot(x_fft, y0lowfft)


plt.show()